# NLP Block (Explanation & Advice Layer)

## Purpose

Convert raw model output (price prediction, damage severity) into plain, trustworthy advice for used-car buyers by explaining what the specs mean and why a price is fair.

## Approach (three stages)
1. **Extract** — LLM turns the listing into strict JSON specs for the price model.
2. **Explain** — LLM turns the predicted price and specs into plain advice with uncertainty.
3. **RAG** — retrieve relevant facts from `carknowledge.md` with embeddings to ground the advice.

**Comparison:** prompt-only vs. RAG-grounded explanation.

## Integration

- **Input received:** predicted price (ML block), damage severity (CV block), listing text (user).
- **Output provided:** the final natural-language explanation shown to the user in the app.

## Data & Tools

- Text source: `carknowledge.md` (used-car buying knowledge base).
- LLM: OpenAI API (`LLM_API_KEY`, `LLM_MODEL` via environment/secrets).
- Embeddings: OpenAI `text-embedding-3-small` + cosine similarity.

In [10]:
# ── Standard library ──
import os
import re
import json
from pathlib import Path

# ── Third-party ──
import numpy as np
import joblib
from openai import OpenAI

## Setup — Configuration, API Client, and Inputs

Load the API configuration, the trained price model (`price_model.pkl` from the ML block),
and the car-knowledge base (`carknowledge.md`). The OpenAI client is created once and reused.

In [11]:
from dotenv import load_dotenv
load_dotenv()   # reads .env from the project root into environment variables
print(".env loaded")

.env loaded


In [12]:
# ── Configuration 
LLM_API_KEY = os.getenv("LLM_API_KEY", " ")
LLM_MODEL   = os.getenv("LLM_MODEL", "gpt-5.4-mini")
EMBED_MODEL = os.getenv("embed-model", "text-embedding-3-small")

# ── Paths ──
MODEL_PATH  = Path("../models/price_model.pkl")
KNOWLEDGE_PATH = Path("../data/carknowledge.md")

# ── OpenAI client ──
client = OpenAI(api_key=LLM_API_KEY) if LLM_API_KEY else None
if client is None:
    print("WARNING: LLM_API_KEY not set — set it before running the LLM cells.")
else:
    print("OpenAI client ready. Model:", LLM_MODEL)

# ── Load the trained price model (output of the ML block) ──
if MODEL_PATH.exists():
    price_bundle = joblib.load(MODEL_PATH)
    print("Price model loaded:", price_bundle.get("model_name", "unknown"))
else:
    price_bundle = None
    print("WARNING: price_model.pkl not found at", MODEL_PATH)

# ── Load the car-knowledge base (text source for RAG) ──
if KNOWLEDGE_PATH.exists():
    knowledge_text = KNOWLEDGE_PATH.read_text(encoding="utf-8")
    print(f"Knowledge base loaded: {len(knowledge_text):,} characters")
else:
    knowledge_text = ""
    print("WARNING: carknowledge.md not found at", KNOWLEDGE_PATH)

OpenAI client ready. Model: gpt-5.4-mini
Price model loaded: HistGradientBoosting
Knowledge base loaded: 23,750 characters


## Stage A — Extract Car Specs from Free Text (LLM)

The user pastes a free-text car listing. An LLM extracts the structured fields the price
model needs and returns them as strict JSON (`temperature=0` for consistency).

In [15]:
import pandas as pd
_df = pd.read_csv("../data/autoscout_clean.csv")
for col in ["fuel_category", "transmission", "body_type"]:
    if col in _df.columns:
        vals = sorted(_df[col].dropna().unique().tolist())
        print(f"{col} ({len(vals)}):")
        print(vals)
        print()

fuel_category (10):
['CNG', 'Diesel', 'Electric', 'Electric/Diesel', 'Electric/Gasoline', 'Ethanol', 'Gasoline', 'LPG', 'Others', 'Unknown']

transmission (4):
['Automatic', 'Manual', 'Semi-automatic', 'Unknown']

body_type (18):
['Box', 'Breakdown truck', 'Car transport', 'Compact', 'Convertible', 'Coupe', 'Flatbed van', 'Flatbed+tarpaulin', 'Hydraulic work platform', 'Off-Road/Pick-up', 'Other', 'Panel van', 'Sedan', 'Station wagon', 'Station wagon/van', 'Transporter', 'Van', 'Van-high roof']



In [16]:
# The price model expects these core spec fields (from autoscout_clean.csv).
EXTRACT_FIELDS = [
    "make", "model", "production_year", "mileage_km",
    "power_hp", "fuel_category", "transmission", "body_type",
]

# Constrain categorical outputs to the EXACT values the price model was trained on.
# This prevents silent encoding mismatches (e.g. "petrol" vs "Gasoline").
FUEL_CATEGORIES = ['CNG', 'Diesel', 'Electric', 'Electric/Diesel', 'Electric/Gasoline',
                   'Ethanol', 'Gasoline', 'LPG', 'Others', 'Unknown']
TRANSMISSIONS   = ['Automatic', 'Manual', 'Semi-automatic', 'Unknown']
BODY_TYPES      = ['Box', 'Breakdown truck', 'Car transport', 'Compact', 'Convertible',
                   'Coupe', 'Flatbed van', 'Flatbed+tarpaulin', 'Hydraulic work platform',
                   'Off-Road/Pick-up', 'Other', 'Panel van', 'Sedan', 'Station wagon',
                   'Station wagon/van', 'Transporter', 'Van', 'Van-high roof']

def extract_specs(listing_text: str) -> dict:
    """Extract structured car specs from a free-text listing using an LLM.
    Categorical fields are constrained to the price model's known vocabulary.
    Missing values are returned as null (no guessing)."""
    if client is None:
        raise RuntimeError("OpenAI client not initialised (LLM_API_KEY missing).")

    system_prompt = (
        "You are a precise assistant that extracts used-car specifications from a "
        "free-text listing. Return ONLY valid JSON, no extra text. "
        f"Use exactly these keys: {', '.join(EXTRACT_FIELDS)}.\n"
        "Rules:\n"
        "- Use numbers for production_year, mileage_km, and power_hp.\n"
        "- 'petrol'/'benzin' maps to 'Gasoline'. Map fuel to ONE of: "
        f"{FUEL_CATEGORIES}.\n"
        "- Map transmission to ONE of: " f"{TRANSMISSIONS}.\n"
        "- Map body_type to the closest ONE of: " f"{BODY_TYPES}.\n"
        "- If a categorical value is stated but unclear, use 'Unknown' (fuel/transmission) "
        "or 'Other' (body_type).\n"
        "- If a value is NOT stated at all, set it to null. Do not guess or invent values."
    )

    example_user = "BMW 320d, 2017, 95'000 km, 190 PS, Diesel, Automat, Limousine."
    example_json = json.dumps({
        "make": "BMW", "model": "320d", "production_year": 2017,
        "mileage_km": 95000, "power_hp": 190, "fuel_category": "Diesel",
        "transmission": "Automatic", "body_type": "Sedan"
    })

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": example_user},
            {"role": "assistant", "content": example_json},
            {"role": "user", "content": listing_text},
        ],
    )
    raw = response.choices[0].message.content
    return json.loads(raw)

In [17]:
# Quick test 
test_listings = [
    "VW Golf 1.6 TDI, Baujahr 2016, 120'000 km, 110 PS, Diesel, manuell, Kombi.",
    "Audi A4 Avant, 2019, 78000 km, 190 hp, petrol, automatic.",
    "Toyota Yaris hybrid 2020, 45'000 km, only 1 owner, great condition.",
]

for i, listing in enumerate(test_listings, 1):
    print(f"--- Listing {i} ---")
    print(listing)
    specs = extract_specs(listing)
    print(json.dumps(specs, indent=2, ensure_ascii=False))
    print()

--- Listing 1 ---
VW Golf 1.6 TDI, Baujahr 2016, 120'000 km, 110 PS, Diesel, manuell, Kombi.
{
  "make": "VW",
  "model": "Golf 1.6 TDI",
  "production_year": 2016,
  "mileage_km": 120000,
  "power_hp": 110,
  "fuel_category": "Diesel",
  "transmission": "Manual",
  "body_type": "Station wagon"
}

--- Listing 2 ---
Audi A4 Avant, 2019, 78000 km, 190 hp, petrol, automatic.
{
  "make": "Audi",
  "model": "A4 Avant",
  "production_year": 2019,
  "mileage_km": 78000,
  "power_hp": 190,
  "fuel_category": "Gasoline",
  "transmission": "Automatic",
  "body_type": "Station wagon"
}

--- Listing 3 ---
Toyota Yaris hybrid 2020, 45'000 km, only 1 owner, great condition.
{
  "make": "Toyota",
  "model": "Yaris hybrid",
  "production_year": 2020,
  "mileage_km": 45000,
  "power_hp": null,
  "fuel_category": "Unknown",
  "transmission": null,
  "body_type": null
}



## Stage B: Explain the Price in Plain, Persona-Adapted Language

Given the extracted specs and the price predicted by the ML model, a second LLM call
produces a short, plain-language explanation for a non-expert buyer. The explanation is
adapted to a buyer persona and always includes one uncertainty/limitation note. This LLM
does NOT compute a price. It only explains the value already produced by the model.

In [18]:
# Buyer personas for adapting the explanation's tone and focus.
PERSONAS = {
    "first_time":  "a first-time car buyer who does not know technical car terms; "
                   "explain jargon simply.",
    "budget":      "a budget-conscious buyer; focus on running costs, reliability, and "
                   "whether the price is fair for the money.",
    "expat":       "a non-native English speaker; use very clear, simple language and "
                   "short sentences.",
}

def generate_explanation(specs: dict, predicted_price: float, persona: str = "first_time") -> str:
    """Explain an already-computed price prediction in plain, persona-adapted language.
    Does NOT compute a new price. Returns a short text with one uncertainty note."""
    if client is None:
        raise RuntimeError("OpenAI client not initialised (LLM_API_KEY missing).")

    persona_desc = PERSONAS.get(persona, PERSONAS["first_time"])

    system_prompt = (
        "You are a helpful used-car buying advisor. "
        f"You are explaining to {persona_desc} "
        "You are given the car's specifications and a price already estimated by a "
        "machine-learning model. Do NOT calculate a new price; only explain the given one.\n"
        "Write a short explanation (3-4 sentences) that:\n"
        "- states the estimated fair price,\n"
        "- explains in plain language what the key specs (mileage, age, fuel, power) mean "
        "for this car's value,\n"
        "- includes exactly one clear uncertainty or limitation.\n"
        "Return ONLY valid JSON with a single key 'answer'."
    )

    user_content = (
        f"Specifications: {json.dumps(specs, ensure_ascii=False)}\n"
        f"Estimated fair price (EUR): {predicted_price:.0f}"
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.3,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
    )
    raw = response.choices[0].message.content
    return json.loads(raw)["answer"]

In [22]:
CURRENT_YEAR = 2026

# Default values for features a free-text listing usually does not provide.
# Categorical OHE columns must get a valid string ("Unknown"/"Other"), never NaN.
NUMERIC_DEFAULTS = {
    "ratings_average": np.nan, "ratings_count": np.nan,
    "ratings_recommend_percentage": np.nan,
    "nr_seats": 5, "nr_doors": 4, "gears": np.nan,
    "cylinders": np.nan, "cylinders_volume_cc": np.nan, "weight_kg": np.nan,
    "fuel_cons_comb_l100_km": np.nan, "co2_emission_grper_wltp_km": np.nan,
    "nr_prev_owners": np.nan,
}
CATEGORICAL_DEFAULTS = {
    "vehicle_type": "Unknown", "body_color": "Unknown", "paint_type": "Unknown",
    "upholstery": "Unknown", "upholstery_color": "Unknown", "drive_train": "Unknown",
    "primary_fuel": "Unknown", "envir_standard": "Unknown",
    "country_code": "Unknown", "seller_type": "Unknown",
}
REMAINDER_DEFAULTS = {  # boolean/flag columns the model passes through untouched
    "has_particle_filter": False, "is_used": True, "is_new": False,
    "is_preregistered": False, "had_accident": False,
    "has_full_service_history": False, "non_smoking": False,
    "is_rental": False, "seller_is_dealer": True,
}
FREQ_DEFAULT_TEXT = {"original_market": "Unknown"}  # freq-encoded, needs a string before mapping

def predict_price(specs: dict) -> float:
    """Predict a fair price (EUR) from extracted specs.
    Builds the full feature row the model expects, fills sensible defaults for
    fields not present in a listing, and computes engineered features."""
    if price_bundle is None:
        raise RuntimeError("Price model not loaded.")

    row = dict(specs)  # start from what the LLM extracted

    # Engineered features (computed the same way as in training)
    year = row.get("production_year")
    row["car_age"] = max(CURRENT_YEAR - year, 0) if year else np.nan
    mileage = row.get("mileage_km")
    if mileage is not None and row.get("car_age") and row["car_age"] > 0:
        row["mileage_per_year"] = mileage / row["car_age"]
    else:
        row["mileage_per_year"] = mileage if mileage is not None else np.nan

    # Fill defaults for everything the listing did not provide
    for col, default in {**NUMERIC_DEFAULTS, **CATEGORICAL_DEFAULTS,
                         **REMAINDER_DEFAULTS, **FREQ_DEFAULT_TEXT}.items():
        if col not in row or row[col] is None:
            row[col] = default

    # Any remaining expected column still missing -> safe default
    for col in price_bundle["feature_cols"]:
        if col not in row or row[col] is None:
            row[col] = "Unknown" if col in price_bundle["preprocessor"].transformers_[1][2] else np.nan

    df_row = pd.DataFrame([row])

    # Frequency-encode high-cardinality columns (unseen -> 0)
    for col, freq in price_bundle["freq_maps"].items():
        df_row[col] = df_row[col].map(freq).fillna(0.0)

    # Order columns exactly as the model expects, then transform + predict
    df_row = df_row[price_bundle["feature_cols"]]
    X = price_bundle["preprocessor"].transform(df_row)
    log_pred = price_bundle["model"].predict(X)[0]
    return float(np.expm1(log_pred))

# Re-run the end-to-end test
listing = "Audi A4 Avant, 2019, 78000 km, 190 hp, petrol, automatic."
specs = extract_specs(listing)
price = predict_price(specs)
print("Specs:", json.dumps(specs, ensure_ascii=False))
print(f"Predicted price: EUR {price:,.0f}")

explanation = generate_explanation(specs, price, persona="first_time")
print("\nExplanation:\n", explanation)

Specs: {"make": "Audi", "model": "A4 Avant", "production_year": 2019, "mileage_km": 78000, "power_hp": 190, "fuel_category": "Gasoline", "transmission": "Automatic", "body_type": "Station wagon"}
Predicted price: EUR 25,282

Explanation:
 The estimated fair price for this 2019 Audi A4 Avant is about €25,282. Its 78,000 km mileage means it has been driven a moderate amount, which lowers the value a bit compared with a lower-mileage car, while being a 2019 model keeps it fairly modern. The 190 hp gasoline engine and automatic transmission usually make it more desirable and comfortable to drive, and the station wagon body style can add practicality for families or luggage. One limitation is that this estimate does not fully account for the car’s condition, service history, or local market demand.


In [23]:
# Verify that the buyer persona meaningfully changes the explanation.
listing = "Audi A4 Avant, 2019, 78000 km, 190 hp, petrol, automatic."
specs = extract_specs(listing)
price = predict_price(specs)
print(f"Car: {specs['make']} {specs['model']} | Predicted price: EUR {price:,.0f}\n")

for persona in ["first_time", "budget", "expat"]:
    print(f"=== Persona: {persona} ===")
    print(generate_explanation(specs, price, persona=persona))
    print()

Car: Audi A4 Avant | Predicted price: EUR 25,282

=== Persona: first_time ===
The estimated fair price for this 2019 Audi A4 Avant is about €25,282. Its 78,000 km mileage means it has been used a fair amount, which usually lowers value compared with a lower-mileage car, while the 2019 age keeps it relatively modern. The 190 hp gasoline engine and automatic transmission are attractive features that can support the price, and the station wagon body style is often practical for families or cargo. One limitation is that this estimate does not account for the car’s exact condition, service history, or any optional equipment.

=== Persona: budget ===
The estimated fair price is about €25,282. This 2019 Audi A4 Avant has moderate mileage at 78,000 km, so it’s not especially low-mileage but still reasonable for its age; the 190 hp gasoline engine and automatic transmission suggest a comfortable, fairly strong daily driver, though running costs will usually be higher than for a smaller car. As 

## Stage C: Grounded Explanation with Retrieval-Augmented Generation (RAG)

The plain explanation relies on the model's own knowledge, which can be vague or inaccurate
on specifics. To ground the advice in verified information, a curated car-knowledge base is
split into concept sections, embedded once, and stored. At query time the relevant
sections are retrieved and injected into the explanation prompt.

The knowledge base is built offline (chunk → embed → store) retrieval and generation run at
query time.

In [24]:
def chunk_by_section(text: str) -> list[str]:
    """Split the knowledge base into concept-section chunks.
    Each chunk is one self-contained topic (heading + its body)."""
    # Split on markdown headings (## or **Bold concept** style); keep the heading with its body.
    raw_parts = re.split(r'\n(?=#{1,3}\s|\*\*[A-Z])', text)
    chunks = [p.strip() for p in raw_parts if len(p.strip()) > 50]  # drop tiny fragments
    return chunks

chunks = chunk_by_section(knowledge_text)
print(f"Created {len(chunks)} chunks.\n")
for i, c in enumerate(chunks):
    preview = c.replace("\n", " ")[:90]
    print(f"[{i}] {preview}...")

Created 11 chunks.

[0] **Used Car Buying Wiki: Essential Concepts and Terminology**...
[1] **Mileage (Odometer Reading)** *   **Definition:** The total distance a vehicle has travel...
[2] **Vehicle History Report (VIN Report)** *   **Definition:** A comprehensive document gener...
[3] **Pre-Purchase Inspection (PPI)** *   **Definition:** A thorough evaluation of a vehicle's...
[4] **Frame Damage (Structural Damage)** *   **Definition:** Severe damage to the vehicle's un...
[5] **Flood Damage** *   **Definition:** Damage caused when a vehicle is partially or fully su...
[6] **Transmission Slipping and Delayed Engagement** *   **Definition:** Common warning signs ...
[7] **State of Health (SOH) — *For Electric Vehicles (EVs)*** *   **Definition:** A metric tha...
[8] **Used Car Lemon Laws** *   **Definition:** State-specific laws that provide legal protect...
[9] ## Mileage  Mileage is key when buying a used car. Our guide explains what qualifies as go...
[10] ## What is the National M

In [25]:
def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed a list of texts with the OpenAI embedding model.
    Returns an array of shape (n_texts, embedding_dim)."""
    if client is None:
        raise RuntimeError("OpenAI client not initialised (LLM_API_KEY missing).")
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in response.data])

# Build the knowledge-base embedding index once (offline step).
chunk_embeddings = embed_texts(chunks)
print(f"Embedded {chunk_embeddings.shape[0]} chunks, dimension {chunk_embeddings.shape[1]}.")

Embedded 11 chunks, dimension 1536.


In [26]:
def cosine_similarity(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Cosine similarity between one query vector and each row of a matrix."""
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-10)
    matrix_norm = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-10)
    return matrix_norm @ query_norm

def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Retrieve the top_k most relevant knowledge chunks for a query."""
    query_vec = embed_texts([query])[0]
    scores = cosine_similarity(query_vec, chunk_embeddings)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_idx]

# Quick test: what does the system retrieve for a mileage-related question?
test_query = "What does high mileage mean for a used car and is it a problem?"
hits = retrieve(test_query, top_k=3)
print(f"Query: {test_query}\n")
for i, h in enumerate(hits, 1):
    print(f"--- Retrieved chunk {i} ---")
    print(h.replace("\n", " ")[:200], "...\n")

Query: What does high mileage mean for a used car and is it a problem?

--- Retrieved chunk 1 ---
## Mileage  Mileage is key when buying a used car. Our guide explains what qualifies as good mileage.  When buying a used car, there’s lots to consider: from body style and fuel type to colour and gea ...

--- Retrieved chunk 2 ---
**Mileage (Odometer Reading)** *   **Definition:** The total distance a vehicle has traveled during its lifetime, as recorded by its odometer. *   **Explanation:** Mileage is a primary indicator of a  ...

--- Retrieved chunk 3 ---
**Used Car Lemon Laws** *   **Definition:** State-specific laws that provide legal protections and remedies (such as a free repair or a refund) to consumers who purchase a defective used vehicle. *    ...



In [27]:
def generate_explanation_rag(specs: dict, predicted_price: float,
                             persona: str = "first_time", top_k: int = 3) -> dict:
    """Explain the predicted price using advice grounded in retrieved knowledge.
    Returns the explanation text plus the retrieved sources used (for transparency)."""
    if client is None:
        raise RuntimeError("OpenAI client not initialised (LLM_API_KEY missing).")

    persona_desc = PERSONAS.get(persona, PERSONAS["first_time"])

    # Build a retrieval query from the car's key characteristics
    query = (f"used car buying advice: {specs.get('mileage_km')} km, "
             f"{specs.get('production_year')}, {specs.get('fuel_category')} engine, "
             f"what to check before buying")
    retrieved = retrieve(query, top_k=top_k)
    context = "\n\n".join(retrieved)

    system_prompt = (
        "You are a helpful used-car buying advisor. "
        f"You are explaining to {persona_desc} "
        "Use ONLY the factual knowledge provided in the context below to support your "
        "advice; do not invent facts. You are given a price already estimated by a "
        "machine-learning model — do NOT calculate a new price, only explain it.\n\n"
        f"CONTEXT (verified car-buying knowledge):\n{context}\n\n"
        "Write a short explanation (3-4 sentences) that states the estimated price, explains "
        "what the key specs mean for value using the context where relevant, and includes "
        "exactly one uncertainty or limitation. Return ONLY valid JSON with key 'answer'."
    )
    user_content = (
        f"Specifications: {json.dumps(specs, ensure_ascii=False)}\n"
        f"Estimated fair price (EUR): {predicted_price:.0f}"
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.3,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
    )
    answer = json.loads(response.choices[0].message.content)["answer"]
    return {"answer": answer, "sources": retrieved}

## Stage C Evaluation: Prompt-Only vs. RAG-Grounded Explanation

The two explanation approaches are compared on the same car. The prompt-only version relies on the model's internal knowledge. The RAG version is grounded in retrieved car-buying knowledge. The comparison examines factual specificity, use of verified concepts, and faithfulness to the knowledge base.

In [28]:
# Compare prompt-only vs. RAG-grounded explanation on the same car.
listing = "Audi A4 Avant, 2019, 78000 km, 190 hp, petrol, automatic."
specs = extract_specs(listing)
price = predict_price(specs)
print(f"Car: {specs['make']} {specs['model']} | Predicted price: EUR {price:,.0f}\n")

print("=== A) PROMPT-ONLY ===")
print(generate_explanation(specs, price, persona="first_time"))
print()

print("=== B) RAG-GROUNDED ===")
rag_result = generate_explanation_rag(specs, price, persona="first_time")
print(rag_result["answer"])
print()
print("Retrieved sources used:")
for i, s in enumerate(rag_result["sources"], 1):
    print(f"  [{i}] {s.splitlines()[0][:70]}")

Car: Audi A4 Avant | Predicted price: EUR 25,282

=== A) PROMPT-ONLY ===
The estimated fair price for this 2019 Audi A4 Avant is about €25,282. Its 78,000 km mileage means it has been used a moderate amount, and being a 2019 model makes it a relatively recent car, both of which help support its value. The gasoline engine with 190 hp suggests a good balance of everyday performance and running costs, while the automatic transmission and station wagon body style can make it more practical and appealing to buyers. One limitation is that the price can still change depending on the car’s condition, service history, and local market demand.

=== B) RAG-GROUNDED ===
The estimated fair price for this 2019 Audi A4 Avant is EUR 25,282. Its 78,000 km mileage is a key value factor because mileage affects a car’s condition, value, and longevity, and higher mileage usually means more wear on important parts. The automatic transmission and station wagon body style can make it appealing for everyday us